# 01 — Exploratory Data Analysis
Understanding the Olist dataset before modelling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
engine = create_engine('sqlite:///../data/ecommerce.db')
print("Connected to warehouse")

## 1. Dataset Overview

In [ ]:
tables = ['dim_customer','dim_product','dim_seller','dim_date','fact_orders']
for t in tables:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', engine)['n'][0]
    print(f'{t:<25} {n:>8,} rows')

## 2. Order Status Distribution

In [ ]:
status = pd.read_sql("""
    SELECT order_status, COUNT(*) AS count
    FROM fact_orders GROUP BY order_status ORDER BY count DESC
""", engine)

fig, ax = plt.subplots(figsize=(9,4))
ax.barh(status['order_status'], status['count'], color='steelblue')
ax.bar_label(ax.containers[0], fmt='%d', padding=4)
ax.set_title('Order Status Distribution')
ax.set_xlabel('Orders')
plt.tight_layout(); plt.show()

## 3. Revenue Trend Over Time

In [ ]:
monthly = pd.read_sql("""
    SELECT strftime('%Y-%m', order_date) AS month,
           SUM(payment_value) AS revenue,
           COUNT(DISTINCT order_id) AS orders
    FROM fact_orders
    WHERE order_status='delivered' AND order_date IS NOT NULL
    GROUP BY month ORDER BY month
""", engine)

fig, ax1 = plt.subplots(figsize=(13,5))
ax2 = ax1.twinx()
ax1.fill_between(monthly['month'], monthly['revenue']/1e3, alpha=0.4, color='#2980b9')
ax1.plot(monthly['month'], monthly['revenue']/1e3, color='#2980b9', lw=2, label='Revenue (R$ 000s)')
ax2.plot(monthly['month'], monthly['orders'], color='#e74c3c', lw=2, linestyle='--', label='Orders')
ax1.set_ylabel('Revenue (R$ thousands)'); ax2.set_ylabel('Order count')
ax1.set_title('Monthly Revenue & Order Volume')
plt.xticks(rotation=45)
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines])
plt.tight_layout(); plt.show()

## 4. Top Categories by Revenue

In [ ]:
cats = pd.read_sql("""
    SELECT p.category_english AS category,
           SUM(f.payment_value) AS revenue,
           COUNT(DISTINCT f.order_id) AS orders,
           AVG(f.review_score) AS avg_review
    FROM fact_orders f JOIN dim_product p ON f.product_id=p.product_id
    WHERE f.order_status='delivered' AND p.category_english IS NOT NULL
    GROUP BY category ORDER BY revenue DESC LIMIT 15
""", engine)

fig, axes = plt.subplots(1,2, figsize=(16,6))
axes[0].barh(cats['category'][::-1], cats['revenue'][::-1]/1e3, color='#2ecc71')
axes[0].set_title('Revenue by Category (R$ 000s)'); axes[0].set_xlabel('Revenue (R$ 000s)')
sc = axes[1].scatter(cats['avg_review'], cats['revenue']/1e3, s=cats['orders']/cats['orders'].max()*800,
                     c=range(len(cats)), cmap='viridis', alpha=0.8)
for _, r in cats.iterrows():
    axes[1].annotate(r['category'][:12], (r['avg_review'], r['revenue']/1e3), fontsize=7)
axes[1].set_xlabel('Avg Review Score'); axes[1].set_ylabel('Revenue (R$ 000s)')
axes[1].set_title('Review Score vs Revenue')
plt.tight_layout(); plt.show()

## 5. Customer Purchase Distribution

In [ ]:
orders_per_cust = pd.read_sql("""
    SELECT c.customer_unique_id, COUNT(DISTINCT f.order_id) AS orders
    FROM fact_orders f JOIN dim_customer c ON f.customer_id=c.customer_id
    WHERE f.order_status='delivered'
    GROUP BY c.customer_unique_id
""", engine)

fig, axes = plt.subplots(1,2,figsize=(14,5))
vc = orders_per_cust['orders'].value_counts().sort_index().head(10)
axes[0].bar(vc.index.astype(str), vc.values, color='#9b59b6')
axes[0].set_title('Orders per Customer'); axes[0].set_xlabel('Number of Orders'); axes[0].set_ylabel('Customers')
axes[1].hist(orders_per_cust['orders'].clip(upper=10), bins=10, color='#e67e22', edgecolor='white')
axes[1].set_title('Distribution (clipped at 10)'); axes[1].set_xlabel('Orders')
repeat = (orders_per_cust['orders'] > 1).sum()
total  = len(orders_per_cust)
print(f"Repeat buyers: {repeat:,} / {total:,} ({repeat/total*100:.1f}%)")
plt.tight_layout(); plt.show()

## 6. Delivery Time Analysis

In [ ]:
delivery = pd.read_sql("""
    SELECT delivery_days, is_late_delivery, customer_id
    FROM fact_orders
    WHERE order_status='delivered' AND delivery_days IS NOT NULL AND delivery_days BETWEEN 1 AND 60
""", engine)

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].hist(delivery['delivery_days'], bins=40, color='#1abc9c', edgecolor='white')
axes[0].axvline(delivery['delivery_days'].median(), color='red', lw=2, label=f"Median: {delivery['delivery_days'].median():.0f} days")
axes[0].set_title('Delivery Time Distribution'); axes[0].set_xlabel('Days'); axes[0].legend()
late_rate = delivery['is_late_delivery'].mean()*100
axes[1].pie([100-late_rate, late_rate], labels=['On Time','Late'],
            colors=['#2ecc71','#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[1].set_title(f'Late Delivery Rate: {late_rate:.1f}%')
plt.tight_layout(); plt.show()

## 7. Payment Type Analysis

In [ ]:
pay = pd.read_sql("""
    SELECT payment_type, COUNT(*) AS orders, SUM(payment_value) AS revenue,
           AVG(installments) AS avg_installments
    FROM fact_orders WHERE order_status='delivered'
    GROUP BY payment_type ORDER BY orders DESC
""", engine)
print(pay.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(pay['payment_type'], pay['revenue']/1e3, color=['#3498db','#e74c3c','#2ecc71','#f39c12'])
ax.set_title('Revenue by Payment Type'); ax.set_ylabel('Revenue (R$ 000s)')
plt.tight_layout(); plt.show()

## 8. Geographic Revenue Heatmap

In [ ]:
geo = pd.read_sql("""
    SELECT c.state, SUM(f.payment_value) AS revenue, COUNT(DISTINCT c.customer_unique_id) AS customers
    FROM fact_orders f JOIN dim_customer c ON f.customer_id=c.customer_id
    WHERE f.order_status='delivered'
    GROUP BY c.state ORDER BY revenue DESC
""", engine)

fig, ax = plt.subplots(figsize=(10,6))
bars = ax.barh(geo['state'][::-1], geo['revenue'][::-1]/1e3, color='#2980b9', alpha=0.85)
ax.bar_label(bars, fmt='R$%.0fK', padding=3, fontsize=8)
ax.set_title('Revenue by Brazilian State'); ax.set_xlabel('Revenue (R$ 000s)')
plt.tight_layout(); plt.show()
print(geo.head(5).to_string(index=False))